In [2]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)


In [3]:
BASE = Path.cwd()
DATA = BASE / "data"
RAW = DATA / "raw"
INTERIM = DATA / "interim"

# ClinVar
dfA = pd.read_csv(f'{INTERIM}/clinvar_1.csv',  low_memory=False)

# dbNSFP
dfB = pd.read_csv(f'{INTERIM}/cleaned_data_SE.csv', low_memory=False)

print("Dataset A shape:", dfA.shape)
print("Dataset B shape:", dfB.shape)

Dataset A shape: (10356, 45)
Dataset B shape: (6673, 30)


In [4]:
# --- Normalize chromosome format ---
dfA['Chromosome'] = dfA['Chromosome'].astype(str).str.replace('chr', '', regex=False)
dfB['#chr'] = dfB['#chr'].astype(str).str.replace('chr', '', regex=False)

In [5]:
dfA['PositionVCF'] = pd.to_numeric(dfA['PositionVCF'], errors='coerce')
dfB['pos(1-based)'] = pd.to_numeric(dfB['pos(1-based)'], errors='coerce')

In [6]:

# --- Drop rows with invalid positions ---
dfA = dfA.dropna(subset=['PositionVCF'])
dfB = dfB.dropna(subset=['pos(1-based)'])

dfA['PositionVCF'] = dfA['PositionVCF'].astype(int)
dfB['pos(1-based)'] = dfB['pos(1-based)'].astype(int)

# --- Normalize alleles ---
dfA['ReferenceAlleleVCF'] = dfA['ReferenceAlleleVCF'].str.upper()
dfA['AlternateAlleleVCF'] = dfA['AlternateAlleleVCF'].str.upper()

dfB['ref'] = dfB['ref'].str.upper()
dfB['alt'] = dfB['alt'].str.upper()


In [7]:
dfA_subset = dfA[[
    'Chromosome',
    'PositionVCF',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
    'label',
    'ReviewStatus',
    'ClinSigSimple'
]].copy()

print("Subset A shape:", dfA_subset.shape)

Subset A shape: (10356, 7)


In [8]:
df_merged = dfB.merge(
    dfA_subset,
    left_on=['#chr', 'pos(1-based)', 'ref', 'alt'],
    right_on=['Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF'],
    how='left'
)

print("Merged shape:", df_merged.shape)

Merged shape: (6673, 37)


In [9]:
df_merged = df_merged.drop(columns=[
    'Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF'
])
print("Merged shape:", df_merged.shape)

Merged shape: (6673, 33)


In [10]:
df_merged.to_csv('merged_dataset.csv', index=False)

print("✅ Merging complete. File saved as 'merged_dataset.csv'")

✅ Merging complete. File saved as 'merged_dataset.csv'
